In [3]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
train = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
test_features = [c for c in test_df.columns if c not in ['week_churn', 'month_churn', 'nid', 'report_date']]

In [5]:
target = 'week_churn'

drop_cols = ['month_churn', 'nid', 'report_date']
features = [c for c in train.columns if c not in drop_cols + [target]]

X = train[features]
y = train[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    eval_metric='AUC',
    loss_function='Logloss',
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100,
    class_weights=[1, 9],
)

model.fit(train_pool, eval_set=val_pool)

y_pred_proba = model.predict_proba(X_val)[:, 1]

# ROC-AUC
roc_auc = roc_auc_score(y_val, y_pred_proba)
print(f"ROC-AUC на валидации: {roc_auc:.4f}")


0:	test: 0.7558523	best: 0.7558523 (0)	total: 326ms	remaining: 5m 25s
100:	test: 0.8137709	best: 0.8137709 (100)	total: 23.7s	remaining: 3m 31s
200:	test: 0.8199648	best: 0.8199732 (199)	total: 45.2s	remaining: 2m 59s
300:	test: 0.8247334	best: 0.8247334 (300)	total: 1m 9s	remaining: 2m 42s
400:	test: 0.8282673	best: 0.8282673 (400)	total: 1m 29s	remaining: 2m 13s
500:	test: 0.8302004	best: 0.8302004 (500)	total: 1m 49s	remaining: 1m 48s
600:	test: 0.8317688	best: 0.8317688 (600)	total: 2m 8s	remaining: 1m 25s
700:	test: 0.8328823	best: 0.8328823 (700)	total: 2m 29s	remaining: 1m 3s
800:	test: 0.8339032	best: 0.8339032 (800)	total: 2m 50s	remaining: 42.3s
900:	test: 0.8347166	best: 0.8347166 (900)	total: 3m 11s	remaining: 21s
999:	test: 0.8353382	best: 0.8353382 (999)	total: 3m 30s	remaining: 0us

bestTest = 0.8353382097
bestIteration = 999

ROC-AUC на валидации: 0.8353


In [6]:
target = 'month_churn'

drop_cols = ['week_churn', 'nid', 'report_date']
features = [c for c in train.columns if c not in drop_cols + [target]]

X = train[features]
y = train[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

model_month = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    eval_metric='AUC',
    loss_function='Logloss',
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100,
    class_weights=[1, 9],
)

model_month.fit(train_pool, eval_set=val_pool)

y_pred_proba = model_month.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, y_pred_proba)
print(f"ROC-AUC на валидации: {roc_auc:.4f}")

0:	test: 0.7939080	best: 0.7939080 (0)	total: 209ms	remaining: 3m 28s
100:	test: 0.8474295	best: 0.8474295 (100)	total: 20.6s	remaining: 3m 2s
200:	test: 0.8597762	best: 0.8597762 (200)	total: 41s	remaining: 2m 43s
300:	test: 0.8703453	best: 0.8703453 (300)	total: 1m	remaining: 2m 19s
400:	test: 0.8792146	best: 0.8792146 (400)	total: 1m 18s	remaining: 1m 57s
500:	test: 0.8866104	best: 0.8866104 (500)	total: 1m 37s	remaining: 1m 36s
600:	test: 0.8924160	best: 0.8924160 (600)	total: 1m 56s	remaining: 1m 17s
700:	test: 0.8973389	best: 0.8973389 (700)	total: 2m 17s	remaining: 58.6s
800:	test: 0.9012893	best: 0.9012893 (800)	total: 2m 38s	remaining: 39.3s
900:	test: 0.9049754	best: 0.9049754 (900)	total: 2m 56s	remaining: 19.4s
999:	test: 0.9086509	best: 0.9086509 (999)	total: 3m 16s	remaining: 0us

bestTest = 0.9086509008
bestIteration = 999

ROC-AUC на валидации: 0.9087


In [8]:
test_features = [c for c in test_df.columns if c not in ['week_churn', 'month_churn', 'nid', 'report_date']]

In [12]:
X_test = test_df[test_features]

test_pred_proba = model.predict_proba(X_test)[:, 1]
submission = test_df[['nid']]
submission['week_churn'] = test_pred_proba

/tmp/ipykernel_60143/1455027837.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission['week_churn'] = test_pred_proba


In [13]:
X_test = test_df[test_features]

test_pred_proba = model_month.predict_proba(X_test)[:, 1]
submission['month_churn'] = test_pred_proba

/tmp/ipykernel_60143/3262388952.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission['month_churn'] = test_pred_proba


In [14]:
submission

,nid,week_churn,month_churn
0,2378,0.614318,0.079157
1,4004,0.806161,0.671764
2,9151,0.191895,0.086383
3,6701,0.682250,0.064090
4,10771,0.063174,0.025295
...,...,...,...
27432,10597,0.724665,0.427849
27433,8737,0.527067,0.599441
27434,2881,0.755966,0.669225
27435,11067,0.838082,0.664189


In [15]:
submission.to_csv('submission.csv', index=False)